In [2]:
from whitebox_workflows import WbEnvironment
import numpy as np
import math
from operator import itemgetter, attrgetter

In [3]:
wbe = WbEnvironment()

In [14]:
class GridCell:
    def __init__(self, row, col, z, flatIndex):
        self.row = row
        self.col = col
        self.z = z
        self.flatIndex = flatIndex

LnOf2 = 0.693147180559945


In [19]:
cells = [GridCell(row = 1, col = 1, z = 1, flatIndex = 2), GridCell(row = 2, col = 2, z = 0, flatIndex = 2),GridCell(row = 3, col = 3, z = 0, flatIndex = 0)]
for cell in cells:
    print(f"{cell.row}: {cell.col}")

1: 1
2: 2
3: 3


In [20]:
cells = sorted(cells, key=attrgetter('z'))
cells = sorted(cells, key=attrgetter('flatIndex'))
for cell in cells:
    print(f"{cell.row}: {cell.col}")

3: 3
2: 2
1: 1


In [1]:

def WetlandOutletsPFO(demFile:str, wetlandRasFile:str, outputFile: str, isAllowMultiOutlet: bool = False, isUsingPFO: bool = True, pntrFile: str = ""):
    """
    Find Wetland Outlets with PFO

    Parameters
    ----------
    demFile             : DEM file path
    wetlandRasFile      : Wetland file path 
    outputFile          : the column name 
    isAllowMultiOutlet  : If multiple outlets are allowed
    isUsingPFO          : If PFO is used
    pntrFile            : The pointer file path

    """

    try:
        progress, oldProgress, col, row, colN, rowN = 0, 0, 0, 0, 0, 0
        numSolvedCells = 0
        n = 0
        z, zN = 0.0, 0.0
        isPit, isEdgeCell, isWetland, flag = False, False, False, False
        gc = None
        """
        * 7  8  1
        * 6  X  2
        * 5  4  3
        """
        dX = [1, 1, 1, 0, -1, -1, -1, 0]
        dY = [-1, 0, 1, 1, 1, 0, -1, -1]
        backLink = [5, 6, 7, 8, 1, 2, 3, 4]

        wetlandRas = wbe.read_raster(wetlandRasFile)

        # read the input image
        dem = wbe.read_raster(demFile)
        nodata = dem.configs.nodata
        rows = dem.configs.rows
        cols = dem.configs.columns


        dem2D = np.full((rows, cols), nodata)
        output = np.full((rows, cols), nodata)
        flowdir = np.full((rows, cols), 0, dtype = np.unsignedinteger) 
        queue = []
        inQueue = np.full((rows, cols), False, dtype = bool) 

        # initialize the grids
        oldProgress = -1
        flatIndex = 0
        k = 0
        dir = 0
        x, y = 0, 0
        if isUsingPFO:                
            #Loop through cells to find cells that are pit and edge
            for row in range(rows):
                for col in range(cols):
                    z = dem[row, col]
                    dem2D[row, col] = z
                    flowdir[row, col] = 0
                    if z != nodata:
                        isWetland = wetlandRas[row, col] > 0
                        isPit = True
                        isEdgeCell = False
                        for n in range(8):
                            zN = dem[row + dY[n], col + dX[n]]
                            if isPit and zN != nodata and zN < z:
                                isPit = False
                            elif zN == nodata:
                                isEdgeCell = True
                        if isPit and isEdgeCell:
                            queue.append(GridCell(row, col, z, 0))
                            inQueue[row, col] = True
                    else:
                        numSolvedCells += 1

            row = 0
            col = 0

            queue = sorted(queue, key=attrgetter('z'))
            queue = sorted(queue, key=attrgetter('flatIndex'))

            while len(queue) > 0:
                gc = queue.pop(0)
                row = gc.row
                col = gc.col
                flatIndex = gc.flatIndex

                for n in range(8):
                    rowN = row + dY[n]
                    colN = col + dX[n]
                    zN = dem2D[rowN, colN]
                    if zN != nodata and not inQueue[rowN, colN]:
                        numSolvedCells += 1
                        flowdir[rowN, colN] = backLink[n]
                        k = 0
                        if zN == dem2D[row, col]:
                            k = flatIndex + 1
                        queue.append(GridCell(rowN, colN, zN, k))

                        inQueue[rowN, colN] = True

                        if wetlandRas[row, col] != wetlandRas[rowN, colN]:
                            flag = True
                            x = colN
                            y = rowN
                            isReturn = False
                            while flag:
                                # find the downslope neighbour
                                dir = flowdir[y, x] - 1
                                if dir > 0:
                                    if dir > 7:
                                        return
                                    x += dX[dir]
                                    y += dY[dir]
                                    if wetlandRas[y, x] > 0:
                                        if wetlandRas[rowN, colN] == wetlandRas[y, x]:
                                            isReturn = True
                                            flag = False
                                        else:
                                            isReturn = False
                                            flag = False
                                else:
                                    flag = False
                            if not isReturn:
                                output[rowN, colN] = wetlandRas[rowN, colN]
        else:
            pntr = wbe.read_raster(pntrFile)
            c = 0
            for row in range(rows):
                for col in range(cols):
                    if pntr[row, col] > 0 and wetlandRas[row, col] > 0:
                        z = dem[row, col]
                        dem2D[row, col] = z
                        dir = int(pntr[row, col])
                        if dir > 0:
                            c = int(math.log(dir) / LnOf2)
                            if c > 7:
                                print("An unexpected value has "
                                             + "been identified in the pointer "
                                             + "image. This tool requires a "
                                             + "pointer grid that has been "
                                             + "created using either the D8 "
                                             + "or Rho8 tools.")
                                return

                            rowN = row + dX[c]
                            colN = col + dY[c]

                            if wetlandRas[row, col] != wetlandRas[rowN, colN]:
                                flag = True
                                x = colN
                                y = rowN
                                isReturn = False

                                count = 0
                                while flag and count < 100:
                                    # find the downslope neighbour
                                    dir = int(pntr[y, x])
                                    if dir > 0:
                                        c = int(math.log(dir) / LnOf2)
                                        if c > 7:
                                            print("An unexpected value has "
                                                         + "been identified in the pointer "
                                                         + "image. This tool requires a "
                                                         + "pointer grid that has been "
                                                         + "created using either the D8 "
                                                         + "or Rho8 tools.")
                                            return

                                        if wetlandRas[y, x] > 0:
                                            if wetlandRas[row, col] == wetlandRas[y, x]:
                                                isReturn = True
                                                flag = False
                                            else:
                                                isReturn = False
                                                flag = False
                                        x += dX[c]
                                        y += dY[c]
                                    else:
                                        flag = False
                                    count += 1
                                if not isReturn:
                                    output[row, col] = wetlandRas[row, col]

        # Group the outlets and use the lowest one as the outlet
        outputNew = np.full((rows, cols), nodata)
        if isAllowMultiOutlet:
            register = {}
            registerReverse = np.full((rows, cols), int(nodata), dtype=int)
            grpNum = 0
            for row in range(rows):
                for col in range(cols):
                    if output[row, col] > 0:
                        loc = [row, col]
                        num = -1
                        for n in range(8):
                            rowN = row + dY[n]
                            colN = col + dX[n]
                            if output[row, col] == output[rowN, colN]:
                                if registerReverse[rowN, colN] > 0:
                                    num = registerReverse[rowN, colN]
                                    break
                        if num < 0:
                            grpNum += 1
                            register[grpNum] = []
                            num = grpNum
                        registerReverse[row, col] = num
                        register[num].append(loc)

            elevMap = {}
            for lst in register.values():
                elevMap = {}
                for loc in lst:
                    elevMap[dem[loc[0], loc[1]]] = loc
                elev = list(elevMap.keys())
                elev.sort()
                lowest = elev[0]
                outputNew[elevMap[lowest][0], elevMap[lowest][1]] = output[elevMap[lowest][0], elevMap[lowest][1]]

            # For multiple outlets wetland, outlet flow to wetland will not be count
            wetlandOutlets = {}
            suspectCounts = {}
            suspects = []
            for row in range(rows):
                for col in range(cols):
                    if outputNew[row, col] > 0 and wetlandRas[row, col] > 0:
                        id = int(wetlandRas[row, col])
                        if id in wetlandOutlets:
                            wetlandOutlets[id] += 1
                        else:
                            wetlandOutlets[id] = 1

                        dir = flowdir[row, col] - 1
                        if dir > 0:
                            if dir > 7:
                                return

                            rowN = row + dX[dir]
                            colN = col + dY[dir]

                            if wetlandRas[rowN, colN] > 0:
                                loc = [row, col]
                                suspects.append(loc)
                                if id in suspectCounts:
                                    suspectCounts[id] += 1
                                else:
                                    suspectCounts[id] = 1

            for loc in suspects:
                id = int(wetlandRas[loc[0], loc[1]])
                if wetlandOutlets[id] > 1 and wetlandOutlets[id] > suspectCounts[id]:
                    outputNew[loc[0], loc[1]] = nodata
        else:
            register = {}
            for row in range(rows):
                for col in range(cols):
                    if output[row, col] > 0:
                        id = int(output[row, col])
                        loc = [row, col]
                        if id not in register:
                            register[id] = []
                        register[id].append(loc)

            elevMap = {}
            for lst in register.values():
                elevMap = {}
                for loc in lst:
                    elevMap[dem[loc[0], loc[1]]] = loc
                elev = list(elevMap.keys())
                elev.sort()
                lowest = elev[0]
                outputNew[elevMap[lowest][0], elevMap[lowest][1]] = output[elevMap[lowest][0], elevMap[lowest][1]]

        # output the data
        outputRaster = wbe.new_raster(dem.configs)        
        for row in range(rows):
            for col in range(cols):
                z = dem2D[row, col]
                if z != nodata:
                    outputRaster[row, col] = outputNew[row, col]

        wbe.write_raster(outputRaster, outputFile)

    except MemoryError as oe:
        pass
    except Exception as e:
        pass
    finally:
        pass




In [11]:
import numpy as np
import math
from datetime import datetime

def ValueAccumD8(d8_ras_file, value_ras_file,output_ras_file, is_average):
    row, col, x, y = 0, 0, 0, 0
    z, z2 = 0.0, 0.0
    v, v2 = 0.0, 0.0
    i = 0
    dX = [1, 1, 1, 0, -1, -1, -1, 0]
    dY = [-1, 0, 1, 1, 1, 0, -1, -1]
    inflowingVals = [16, 32, 64, 128, 1, 2, 4, 8]
    numInNeighbours = 0.0
    isAvg = False
    flag = False
    flowDir = 0.0

    isAvg = is_average

    try:
        wbe = WbEnvironment()

        pntr = wbe.read_raster(d8_ras_file)
        valueRas = wbe.read_raster(value_ras_file)

        noData = pntr.configs.nodata
        rows = pntr.configs.rows
        cols = pntr.configs.columns
        rowsLessOne = rows - 1
 
        output = wbe.new_raster(pntr.configs)
        temp = np.full((rows, cols), 1.0)

        tmpGrid1 = wbe.new_raster(pntr.configs)

        for row in range(rows):
            for col in range(cols):
                if pntr[row, col] != noData and valueRas[row, col] != noData:
                    z = 0
                    for i in range(8):
                        if pntr[row + dY[i], col + dX[i]] == inflowingVals[i]:
                            z += 1
                    tmpGrid1[row, col] = z
                    output[row, col] = valueRas[row, col]
                else:
                    temp[row, col] = noData

        for row in range(rows):
            for col in range(cols):
                if tmpGrid1[row, col] == 0:
                    tmpGrid1[row, col] = -1
                    x, y = col, row
                    while True:
                        z = temp[y, x]
                        v = output[y, x]
                        flowDir = pntr[y, x]
                        if flowDir > 0:
                            i = int(math.log(flowDir) / math.log(2))
                            x += dX[i]
                            y += dY[i]
                            z2 = temp[y, x]
                            temp[y, x] = z2 + z
                            v2 = output[y, x]
                            output[y, x] = v2 + v
                            numInNeighbours = tmpGrid1[y, x] - 1
                            tmpGrid1[y, x] = numInNeighbours
                            if numInNeighbours == 0:
                                tmpGrid1[y, x] = -1
                                flag = True
                            else:
                                flag = False
                        else:
                            flag = False
                        if not flag:
                            break

        if isAvg:
            for row in range(rows):
                for col in range(cols):
                    flowDir = output[row, col]
                    if flowDir != noData and valueRas[row, col] != noData:
                        output[row, col] = output[row, col] / temp[row, col]
                    else:
                        output[row, col] = noData

        #save here
        wbe.write_raster(output, output_ras_file)

    except MemoryError:
        print("memory error")
    except Exception as e:
        print(e)
    finally:
        pass




In [12]:
d8 = r"C:\Work\imWEBs\Interface\BRC_TIFFs\flow_dir.tif"
prc = r"C:\Work\imWEBs\Interface\BRC_TIFFs\PRC.tif"
output_file = r"C:\Work\imWEBs\Interface\BRC_TIFFs\test\PRC_Acc_Avg_WBE.tif"

ValueAccumD8(d8,prc, output_file, True)